# Dataset audit

Audit of the provided labels and documents **before** any extraction model is built.

**Test-label policy.** Values in `test.csv` are never displayed or used here. Test labels are summarised by
structure and counts only. The `train.csv` row for `24158401-Rental-Agreement` describes a document that lives in
`test/`, so it is treated as a test label too.

Reusable logic lives in `src/audit.py`; this notebook runs it and records the reviewed conclusions.
Prerequisite: `python -m src.extract_text --split all` (cached document text is used for label-support checks).

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from IPython.display import display

from src import audit
from src.config import FILE_NAME_COLUMN, LABEL_COLUMNS, OUTPUTS_DIR, TEST_CSV, TEST_DIR, TRAIN_CSV, TRAIN_DIR
from src.dataset import load_labels

pd.set_option("display.max_colwidth", 80)
report = audit.build_report(TRAIN_CSV, TEST_CSV, TRAIN_DIR, TEST_DIR)
leaked = report["cross_split"]["train_label_rows_for_test_documents"]

## 1. CSV structure: columns, shape, dtypes

In [2]:
overview = pd.DataFrame({
    split: {"rows": report[split]["overview"]["rows"], "columns": len(report[split]["overview"]["columns"])}
    for split in ("train", "test")
}).T
display(overview)
print("Exact column names (identical in both files):", report["train"]["overview"]["columns"])
assert report["train"]["overview"]["columns"] == report["test"]["overview"]["columns"]
display(pd.DataFrame({s: report[s]["overview"]["inferred_dtypes"] for s in ("train", "test")}))

,rows,columns
train,10,7
test,4,7


Exact column names (identical in both files): ['File Name', 'Aggrement Value', 'Aggrement Start Date', 'Aggrement End Date', 'Renewal Notice (Days)', 'Party One', 'Party Two']


,train,test
File Name,object,object
Aggrement Value,int64,int64
Aggrement Start Date,object,object
Aggrement End Date,object,object
Renewal Notice (Days),float64,int64
Party One,object,object
Party Two,object,object


Pandas' default inference gives `Renewal Notice (Days)` as `float64` in train because one cell is blank, and
dates as plain strings. The pipeline therefore reads labels as **raw strings** (`dtype=str`) so nothing is
silently converted. The misspelling `Aggrement` is kept exactly.

### Sample rows (train only, excluding the row that describes a test document)

In [3]:
train = load_labels(TRAIN_CSV)
dev_rows = train[~train[FILE_NAME_COLUMN].str.strip().isin(leaked)]
display(dev_rows.head())

,File Name,Aggrement Value,Aggrement Start Date,Aggrement End Date,Renewal Notice (Days),Party One,Party Two
0,6683127-House-Rental-Contract-GERALDINE-GALINATO-v2-Page-1,6500,20.05.2007,20.05.2008,15,"Antonio Levy S. Ingles, Jr. and/or Mary Rose C. Ingles",GERALDINE Q. GALINATO
1,6683129-House-Rental-Contract-Geraldine-Galinato-v2,6500,20.05.2007,20.05.2008,15,"Antonio Levy S. Ingles, Jr. and/or Mary Rose C. Ingles",GERALDINE Q. GALINATO
2,18325926-Rental-Agreement-1,4000,05.12.2008,31.11.2009,90,MR.K.Kuttan,P.M. Narayana Namboodri
4,36199312-Rental-Agreement,3800,01.05.2010,31.04.2011,30,Balaji.R,Kartheek R
5,44737744-Maddireddy-Bhargava-Reddy-Rental-Agreement,3000,20.09.2010,19.07.2011,,M.V.V. VIJAYA SHANKAR,MADDIREDDY BHARGAVA REDDY


## 2. Document inventory and CSV-to-file matching

In [4]:
for split in ("train", "test"):
    files = report[split]["files"]
    print(f"--- {split}: formats {files['file_formats']}, matched {len(files['matched'])}")
    for key in ("label_without_file", "file_without_label", "duplicate_ids_in_csv"):
        print(f"  {key}: {files[key] or 'none'}")
print("\ntrain.csv rows describing test documents:", leaked)

--- train: formats {'.docx': 7, '.png': 3}, matched 9
  label_without_file: ['24158401-Rental-Agreement']
  file_without_label: ['46239065-Standard-Rental-Agreement-Rental-With-Performance-Fee']
  duplicate_ids_in_csv: none
--- test: formats {'.docx': 2, '.png': 2}, matched 4
  label_without_file: none
  file_without_label: none
  duplicate_ids_in_csv: none

train.csv rows describing test documents: ['24158401-Rental-Agreement']


## 3. Label quality: missing values, formatting, invalid dates

In [5]:
issues = pd.DataFrame(report["train"]["label_issues"])
display(issues.assign(value=issues["value"].map(repr)))

,document,column,issue,value
0,18325926-Rental-Agreement-1,Aggrement End Date,impossible calendar date,'31.11.2009'
1,18325926-Rental-Agreement-1,Party One,leading/trailing whitespace,'MR.K.Kuttan '
2,18325926-Rental-Agreement-1,Party Two,leading/trailing whitespace,'P.M. Narayana Namboodri '
3,36199312-Rental-Agreement,Aggrement End Date,impossible calendar date,'31.04.2011'
4,36199312-Rental-Agreement,Party One,leading/trailing whitespace,'Balaji.R '
5,44737744-Maddireddy-Bhargava-Reddy-Rental-Agreement,Renewal Notice (Days),missing,''
6,47854715-RENTAL-AGREEMENT,Aggrement End Date,impossible calendar date,'31.02.2011'
7,47854715-RENTAL-AGREEMENT,Party One,leading/trailing whitespace,' P C MATHEW'
8,47854715-RENTAL-AGREEMENT,Party Two,leading/trailing whitespace,' L GOPINATH '
9,50070534-RENTAL-AGREEMENT,Aggrement Start Date,leading/trailing whitespace,'01.04.2010 '


In [6]:
print("Test labels, counts only:")
display(pd.DataFrame(report["test"]["label_issue_counts"]).fillna(0).astype(int))
print("Row(s) in train.csv describing test documents, counts only:", report["train"]["leaked_row_issue_counts"])

Test labels, counts only:


,Party One,Party Two
leading/trailing whitespace,4,3
starts with punctuation,0,1


Row(s) in train.csv describing test documents, counts only: {'Party One': {'leading/trailing whitespace': 1}, 'Party Two': {'leading/trailing whitespace': 1}}


Invalid dates are **flagged, not corrected**: the reference CSVs are untouched. A correct extractor cannot
exactly match an impossible date such as `31.02.2011`; this is reported as a dataset limitation in evaluation.

## 4. Duplicates and near-duplicates

In [7]:
print("Train rows with identical labels:", report["train"]["identical_label_rows"])
print("Near-duplicate train documents (word 3-gram containment >= 0.5):")
display(pd.DataFrame(report["train"]["near_duplicate_documents"]))
print("Near-duplicate documents across train/test:", report["cross_split"]["near_duplicate_documents"] or "none")

Train rows with identical labels: [['6683127-House-Rental-Contract-GERALDINE-GALINATO-v2-Page-1', '6683129-House-Rental-Contract-Geraldine-Galinato-v2']]
Near-duplicate train documents (word 3-gram containment >= 0.5):


,a,b,similarity
0,6683127-House-Rental-Contract-GERALDINE-GALINATO-v2-Page-1,6683129-House-Rental-Contract-Geraldine-Galinato-v2,0.959


Near-duplicate documents across train/test: none


## 5. Do the train labels match their own documents?

In [8]:
rows = []
for r in report["train"]["label_support"]:
    rows.append({
        "document": r["document"],
        "party_one_own": r["Party One"]["own_score"],
        "party_two_own": r["Party Two"]["own_score"],
        "party_best_doc": r["Party One"]["best_document"],
        "value_in_own_text": r["Aggrement Value"]["found_in_own_text"],
        "value_found_in": r["Aggrement Value"]["found_in"],
        "fits_other_doc": r["fits_other_document_better"],
    })
display(pd.DataFrame(rows))

,document,party_one_own,party_two_own,party_best_doc,value_in_own_text,value_found_in,fits_other_doc
0,6683127-House-Rental-Contract-GERALDINE-GALINATO-v2-Page-1,1.0,1.0,6683127-House-Rental-Contract-GERALDINE-GALINATO-v2-Page-1,False,[6683129-House-Rental-Contract-Geraldine-Galinato-v2],False
1,6683129-House-Rental-Contract-Geraldine-Galinato-v2,1.0,1.0,6683127-House-Rental-Contract-GERALDINE-GALINATO-v2-Page-1,True,[6683129-House-Rental-Contract-Geraldine-Galinato-v2],False
2,18325926-Rental-Agreement-1,1.0,1.0,18325926-Rental-Agreement-1,True,[18325926-Rental-Agreement-1],False
3,36199312-Rental-Agreement,1.0,1.0,36199312-Rental-Agreement,True,[36199312-Rental-Agreement],False
4,44737744-Maddireddy-Bhargava-Reddy-Rental-Agreement,0.5,1.0,44737744-Maddireddy-Bhargava-Reddy-Rental-Agreement,False,[],False
5,47854715-RENTAL-AGREEMENT,1.0,1.0,47854715-RENTAL-AGREEMENT,True,[47854715-RENTAL-AGREEMENT],False
6,50070534-RENTAL-AGREEMENT,1.0,1.0,50070534-RENTAL-AGREEMENT,True,[50070534-RENTAL-AGREEMENT],False
7,54770958-Rental-Agreement,0.0,0.0,54945838-Rental-Agreement,False,[54945838-Rental-Agreement],True
8,54945838-Rental-Agreement,0.0,0.0,54770958-Rental-Agreement,False,[54770958-Rental-Agreement],True


The labels for `54770958` and `54945838` fit **each other's** document (party names score 0 on their own
text and 0.75–1.0 on the other one; each value is found only in the other file). This was also confirmed by
viewing the scans. The value is not found literally in `6683127` (written `P 6.500.00` / in words) or in
`44737744` (amount is illegible in that DOCX: `Rs 9.99.7.9°`).

## 6. What does `Aggrement Value` mean?

In [9]:
for c in report["train"]["agreement_value_contexts"]:
    print(f"{c['document']}  value={c['value']}")
    for s in c["snippets"][:2]:
        print("    ...", s, "...")

6683127-House-Rental-Contract-GERALDINE-GALINATO-v2-Page-1  value=6500
6683129-House-Rental-Contract-Geraldine-Galinato-v2  value=6500
    ... no due date security deposit in cash 1 5 20 2007 php 6500 00 monthly rental advance payment in cash from to cash 2 5 ...
    ... cash 2 5 20 2007 5 20 2007 6 20 2007 php 6500 00 payment in pdcs from to pdcs 3 6 20 2007 6 ...
18325926-Rental-Agreement-1  value=4000
    ... proprietor of the business and he has agreed to pay rent rs 4000 00 rupees four thousand only for above said premises with the following ...
36199312-Rental-Agreement  value=3800
    ... payable by the lessees to the lessor is a sum of rs 3800 rupees thirty eight thousand only on or before 10th of every english ...
44737744-Maddireddy-Bhargava-Reddy-Rental-Agreement  value=3000
47854715-RENTAL-AGREEMENT  value=9000
    ... the second part for residential use on a monthly rent of rs 9000 rupees nine thousand only plus monthly maintenance expenses levied on by the ...
    ... lea

**Reviewed conclusion (train documents only).**

| Field | What the train labels encode | Evidence |
|---|---|---|
| Aggrement Value | **Monthly rent** (not deposit / advance), digits only, no currency or decimals | `18325926` rent Rs.4,000 vs advance Rs.35,000 → 4000; `36199312` rent Rs.3800 vs deposit Rs.40,000 → 3800; `47854715` rent 9000 vs deposit 90000 → 9000 |
| Value when digits and words disagree | The **digits** | `47854715` "Rs.9000/- (Rupees Seven thousand)" → 9000; `36199312` "Rs.3800/- (Rupees Thirty Eight Thousand)" → 3800 |
| Aggrement Start Date | **Tenancy commencement** date when stated; otherwise the execution date | `47854715` executed 21 Mar 2010, lease from 1 Apr 2010 → 01.04.2010; `50070534` executed 14 Jul 2010, period from 01.04.2010 → 01.04.2010; `18325926` only execution date → 05.12.2008 |
| Aggrement End Date | Stated end date when present; otherwise **derived** from start + term (and the annotators' arithmetic is sometimes invalid) | `50070534` "ending with 30.03.2011"; `47854715` 11 months from 01.04.2010 → `31.02.2011` (impossible) |
| Renewal Notice (Days) | Despite its name, the **termination/vacating notice period**, converted to days (1 month = 30) | `47854715` has *both* "Two months notice … to terminate" and "one month advance notice … for renewal"; label = **60**. `6683129` TERMINATION clause "at least 15 days" → 15 |
| Renewal Notice when absent | Blank | `44737744` has no notice clause and a blank label |
| Party One / Party Two | Lessor/owner first, lessee/tenant second; name only (titles like Mr./Prof. and S/o-relations usually dropped, but not consistently: `MR.K.Kuttan`) | all usable train rows |

### Ambiguities to carry into schema and evaluation design
1. "Renewal Notice" labels follow the termination clause, which conflicts with the field's name.
2. Several end dates are impossible calendar dates (`31.11`, `31.04`, `31.02`): exact match against them is unattainable for a valid date.
3. `6683127` (page 1 only) carries labels copied from the full contract `6683129`; its notice period (15) is not in its own text.
4. `44737744`'s rent is unreadable in the source DOCX; label 3000 is unsupported by the text.
5. Two train label rows are swapped (`54770958` ↔ `54945838`).
6. Party-name formatting (titles, initials, `&` vs `and`) is inconsistent, so raw exact match is strict.
7. With 9 usable train documents (8 distinct contracts), any development metric has very high variance.

## 7. Save the machine-readable audit report

In [10]:
report["reviewed_conclusions"] = {
    "agreement_value": "monthly rent in digits; digits win when digits and words disagree",
    "start_date": "tenancy commencement date when stated, otherwise execution date; DD.MM.YYYY",
    "end_date": "stated end date, otherwise start + term; some reference end dates are impossible calendar dates",
    "renewal_notice_days": "termination/vacating notice period in days (1 month = 30); blank when absent",
    "party_one": "lessor / owner",
    "party_two": "lessee / tenant",
    "swapped_label_rows": ["54770958-Rental-Agreement", "54945838-Rental-Agreement"],
    "labels_not_supported_by_own_text": {
        "6683127-House-Rental-Contract-GERALDINE-GALINATO-v2-Page-1": ["Renewal Notice (Days)"],
        "44737744-Maddireddy-Bhargava-Reddy-Rental-Agreement": ["Aggrement Value"],
    },
}
out = OUTPUTS_DIR / "dataset_audit.json"
audit.save_report(report, out)
print("saved", out.relative_to(PROJECT_ROOT))

saved outputs\dataset_audit.json
